In [15]:
import sys, os
import pandas as pd
import numpy as np
import itertools
from collections import defaultdict
# Notebook uses functions from .py scripts in scripts/ folder
sys.path.append('../../scripts/coordinates_analysis')

from load import load_samples
from compare import compare_samples
from save import save_overlaps

# Mapping replication origins via MCM subunits binding sites


In [5]:
####### LOAD DATA #######
# Load coomon peaks for MCM2 and MCM4
output_dir = '../../results/compared_peaks'
MCM_G1 = pd.read_csv(f'{output_dir}/RDY368_G1_nTr_nT_nC_RDY412_G1_nTr_nT_nC/in_RDY412_G1_nTr_nT_nC_RDY368_G1_nTr_nT_nC_ex.bed', # output from notebooks/compare_coordinates/1_compare_peaks.ipynb
                names=['chr', 'start', 'end', 'center'],
                sep='\t', header=None)

# Save as MCM_G1 in resources/
MCM_G1.to_csv(os.path.join('../../resources', 'MCM_G1.bed'), sep='\t', index=False, header=False)

In [6]:
MCM_G1

,chr,start,end,center
0,chrI,6399,6549,6474
1,chrI,8119,8269,8194
2,chrI,10063,10213,10138
3,chrI,16805,16955,16880
4,chrI,23233,23383,23308
...,...,...,...,...
767,chrXVI,881198,881348,881273
768,chrXVI,909547,909697,909622
769,chrXVI,929661,929811,929736
770,chrXVI,933108,933258,933183


## Annotate mapped origins

In [8]:
ARS_SGD = pd.read_csv('../../resources/ARS_SGD.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
ARS_early = pd.read_csv('../../resources/ARS_early.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
ARS_late = pd.read_csv('../../resources/ARS_late.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)
Sld3_G1 = pd.read_csv('../../../ChEC_Data_Complete/homer_peaks_merged/RDY342_G1_nTr_nT_nC.bed', names=['chr', 'start', 'end', 'center'], sep='\t', header=None)

In [9]:
def check_match(row, factor):
    mask = (factor['chr'] == row['chr']) & \
           (row['center'] + 75 >= factor['start']) & \
           (row['center'] - 75 <= factor['end'])
    return mask.any()

In [21]:
MCM_table = pd.DataFrame()

# 1. Create id for each origin
chr_list = MCM_G1['chr'].to_list()
id_list = []
d = {'chrI':0,'chrII':0,'chrIII':0,'chrIV':0,'chrV':0,'chrVI':0,'chrVII':0,'chrVIII':0,
          'chrIX':0,'chrX':0,'chrXI':0,'chrXII':0,'chrXIII':0,'chrXIV':0,'chrXV':0,'chrXVI':0}
for chromosome in chr_list:
    d[chromosome] += 1
    id_list.append(f'{chromosome}:{d[chromosome]}')
MCM_table['id'] = id_list
MCM_table['chr'] = MCM_G1['chr']

# 2. Get coordinate for origin
MCM_table['center'] = MCM_G1['center']

# 3. Label origin based on overlapping with provided factor
MCM_table['ARS_SGD'] = MCM_table.apply(check_match, axis=1, factor=ARS_SGD).astype(int)
MCM_table['ARS_early'] = MCM_table.apply(check_match, axis=1, factor=ARS_early).astype(int)
MCM_table['ARS_late'] = MCM_table.apply(check_match, axis=1, factor=ARS_late).astype(int)
MCM_table['Sld3_G1'] = MCM_table.apply(check_match, axis=1, factor=Sld3_G1).astype(int)
MCM_table

,id,chr,center,ARS_SGD,ARS_early,ARS_late,Sld3_G1
0,chrI:1,chrI,6474,0,0,0,0
1,chrI:2,chrI,8194,1,0,0,0
2,chrI:3,chrI,10138,0,0,0,0
3,chrI:4,chrI,16880,0,0,0,0
4,chrI:5,chrI,23308,0,0,0,0
...,...,...,...,...,...,...,...
767,chrXVI:66,chrXVI,881273,0,0,0,0
768,chrXVI:67,chrXVI,909622,0,0,0,0
769,chrXVI:68,chrXVI,929736,0,0,0,0
770,chrXVI:69,chrXVI,933183,1,1,1,0


In [22]:
# Save table
MCM_table.to_csv('../../resources/MCM_table.tsv', sep='\t')